Az Ollama fusson, mindkét modellel (`gemma4:e4b`, `bge-m3`). Futtasd sorban: **Shift + Enter**. Ha valami elakad: **Restart**, aztán **Run All**. A Copilot kódját olvasd el, mielőtt **Keep**-et nyomsz; ha rossz, **Undo**.

In [ ]:
import json
import re
import time
from pathlib import Path

import numpy as np
import pandas as pd
import pymupdf

import kurzus
from kurzus import chat, embed, kerdez, utolso_hasznalat
from kurzus import ellenoriz, ellenoriz_feltetel, ellenoriz_fuggveny

## 1. Az index: indítsd el most

5 jelentés (a 4. alkalom négye + a 2024. márciusi, a mintaprojekt miatt), 1219 darab. **Futtasd le most a három cellát**; a beágyazás nálam kb. 85 mp, nálad több is lehet. Az első kettő a 3. és 4. alkalom függvényei (csak futtasd).

In [ ]:
# A 3. alkalom függvényei: PDF → bekezdések → darabok. Csak futtasd.
FEJLEC_JELEK = ("INFLÁCIÓS JELENTÉS •", "MAGYAR NEMZETI BANK")
DARABOK_FAJL = Path("data/cache/darabok.json")


def tisztit(szoveg):
    """PDF-ből kinyert szöveg takarítása: elválasztás, pontsorok, sortörések, szóközök."""
    szoveg = re.sub(r"-\n(?=[a-záéíóöőúüű])", "", szoveg)  # sor végi elválasztás: ren-|delkezésére
    szoveg = re.sub(r"\.{4,}", " ", szoveg)                # tartalomjegyzék pontsora
    szoveg = szoveg.replace("\n", " ")
    return re.sub(r"\s+", " ", szoveg).strip()


def nyomtatott_oldalszam(oldal):
    """Az élőfejben álló nyomtatott oldalszám; ha az oldalon nincs élőfej (borító, belső borító), None."""
    elso_sorok = [s.strip() for s in oldal.get_text().splitlines()[:5]]
    if not any("INFLÁCIÓS JELENTÉS •" in s for s in elso_sorok):
        return None
    for sor in elso_sorok:
        if sor.isdigit():
            return int(sor)
    return None


def oldal_bekezdesei(oldal):
    """Az oldal bekezdései takarítva, élőfej, grafikonfeliratok és oldalszám nélkül."""
    bekezdesek = []
    for blokk in oldal.get_text("blocks"):
        szoveg = tisztit(blokk[4])
        if any(jel in szoveg for jel in FEJLEC_JELEK) or szoveg.isupper():
            continue
        if len(szoveg.split()) < 5 or not re.search(r"[a-zA-ZáéíóöőúüűÁÉÍÓÖŐÚÜŰ]{3}", szoveg):
            continue
        bekezdesek.append(szoveg)
    return bekezdesek


def jelentes_oldalai(pdf_utvonal, jelentes):
    """Egy jelentés oldalai: jelentés, PDF-oldalszám, nyomtatott oldalszám és a bekezdések."""
    return [{"jelentes": jelentes, "pdf_oldal": i, "oldal": nyomtatott_oldalszam(o), "bekezdesek": oldal_bekezdesei(o)}
            for i, o in enumerate(pymupdf.open(pdf_utvonal), start=1)]


def darabol_fix(szoveg, meret=1000, atfedes=0):
    """Fix méretű darabok (karakterben); az egymást követő darabok `atfedes` karakternyit átfednek."""
    lepes = meret - atfedes
    return [szoveg[i:i + meret] for i in range(0, max(len(szoveg) - atfedes, 1), lepes)]


ROVIDITESEK = {"pl", "kb", "stb", "ill", "ún", "vö", "szp", "Kr", "dr"}


def mondatokra(szoveg):
    """Mondatokra bontja a szöveget. Mondatvég: pont, kérdőjel vagy felkiáltójel, utána szóköz és
    nagybetű vagy számjegy; kivéve, ha a pont előtt szám, római szám, egyetlen betű vagy rövidítés áll."""
    mondatok = []
    for resz in re.split(r"(?<=[.!?]) (?=[A-ZÁÉÍÓÖŐÚÜŰ0-9])", szoveg):
        elozo_szo = mondatok[-1].split()[-1].rstrip(".") if mondatok else ""
        if elozo_szo.isdigit() or re.fullmatch(r"[IVX]+|\w", elozo_szo) or elozo_szo in ROVIDITESEK:
            mondatok[-1] += " " + resz
        else:
            mondatok.append(resz)
    return mondatok


def darabol_bekezdesenkent(bekezdesek, max_hossz=1000):
    """Egész bekezdéseket fűz össze, amíg a darab el nem érné a `max_hossz` karaktert. A túl hosszú
    bekezdést mondatokra bontja; ha egyetlen mondat is túl hosszú, azt szavakra."""
    reszek = []
    for bekezdes in bekezdesek:
        if len(bekezdes) <= max_hossz:
            reszek.append(bekezdes)
            continue
        for mondat in mondatokra(bekezdes):
            reszek += [mondat] if len(mondat) <= max_hossz else mondat.split()
    darabok, aktualis = [], ""
    for resz in reszek:
        if aktualis and len(aktualis) + 1 + len(resz) > max_hossz:
            darabok.append(aktualis)
            aktualis = resz
        else:
            aktualis = (aktualis + " " + resz).strip()
    if aktualis:
        darabok.append(aktualis)
    return darabok


def darabok_keszitese(oldalak, max_hossz=1000):
    """Egy jelentés darabjai; minden darab megőrzi a jelentést és mindkét oldalszámot."""
    darabok = []
    for o in oldalak:
        for j, szoveg in enumerate(darabol_bekezdesenkent(o["bekezdesek"], max_hossz)):
            darabok.append({"id": f'{o["jelentes"]}_p{o["pdf_oldal"]:03d}_{j}', "jelentes": o["jelentes"],
                            "pdf_oldal": o["pdf_oldal"], "oldal": o["oldal"], "szoveg": szoveg})
    return darabok


def darabok_betoltese(fajl=DARABOK_FAJL):
    """A forrasok.csv összes jelentésének darabjai a gyorsítótárból; ha nincs meg, újra elkészíti és elmenti (kb. 10 mp)."""
    if fajl.exists():
        return json.loads(fajl.read_text(encoding="utf-8"))
    print("Nincs meg a darabok.json: újra elkészítem a 3. alkalom függvényeivel.")
    darabok = []
    for s in pd.read_csv("data/mnb/forrasok.csv", sep=";").itertuples():
        darabok += darabok_keszitese(jelentes_oldalai(kurzus.letolt(s.url, "data/mnb/" + s.fajl), s.jelentes))
    fajl.parent.mkdir(parents=True, exist_ok=True)
    fajl.write_text(json.dumps(darabok, ensure_ascii=False), encoding="utf-8")
    return darabok

In [ ]:
# A 4. alkalom függvényei: beágyazás, index, keresés, ellenőrzés a nyers PDF-ből. Csak futtasd.
INDEX_FAJL = Path("data/cache/embeddingek_rag.npz")    # saját fájl: más jelentések, mint a 4. alkalommal


def beagyaz(szovegek, adag=64):
    """Szövegek embeddingje adagokban (egy hívás = `adag` szöveg); numpy-mátrixot ad vissza."""
    vektorok = []
    for i in range(0, len(szovegek), adag):
        vektorok += embed(szovegek[i:i + adag])
    return np.array(vektorok, dtype=np.float32)


def index_betoltese(darabok, fajl=INDEX_FAJL):
    """A darabok normalizált embedding-mátrixa: a gyorsítótárból, ha ugyanezekhez a darabokhoz
    készült, különben kiszámolja és elmenti. Az i. sor az i. darabé."""
    idk = [d["id"] for d in darabok]
    if fajl.exists():
        tarolt = np.load(fajl)
        if list(tarolt["idk"]) == idk:
            print(f"Gyorsítótárból: {len(idk)} darab (a kiszámolás {float(tarolt['ido']):.0f} mp volt).")
            return tarolt["vektorok"]
    kezdes = time.perf_counter()
    vektorok = beagyaz([d["szoveg"] for d in darabok])
    vektorok = vektorok / np.linalg.norm(vektorok, axis=1, keepdims=True)   # normalizálás
    ido = time.perf_counter() - kezdes
    fajl.parent.mkdir(parents=True, exist_ok=True)
    np.savez(fajl, vektorok=vektorok, idk=np.array(idk), ido=ido)
    print(f"Kiszámolva: {len(idk)} darab, {ido:.0f} mp ({len(idk) / ido:.1f} darab/mp).")
    return vektorok


def keres(kerdes, darabok, index, k=5):
    """A kérdéshez leghasonlóbb k darab, (hasonlóság, darab) párokként, csökkenő sorrendben."""
    q = np.array(embed(kerdes)[0])
    q = q / np.linalg.norm(q)
    hasonlosagok = index @ q                       # koszinusz-hasonlóság az összes darabbal egyszerre
    legjobbak = np.argsort(-hasonlosagok)[:k]      # a k legnagyobb hasonlóság helye
    return [(float(hasonlosagok[i]), darabok[i]) for i in legjobbak]


def kiir(talalatok):
    """A találatok kiírása: hasonlóság, forrás, a szöveg eleje."""
    for hasonlosag, d in talalatok:
        hol = f'{d["oldal"]}. oldal' if d["oldal"] else f'PDF {d["pdf_oldal"]}. oldal'
        print(f'{hasonlosag:.3f}  {d["jelentes"]}, {hol}:  {d["szoveg"][:140]}…')


forrasok = pd.read_csv("data/mnb/forrasok.csv", sep=";")


def oldalak_ahol(minta, jelentesek):
    """(jelentés, PDF-oldal) párok, ahol a nyers PDF-szövegben megvan a minta (reguláris kifejezés)."""
    talalt = set()
    for s in forrasok[forrasok["jelentes"].isin(jelentesek)].itertuples():
        for i, oldal in enumerate(pymupdf.open("data/mnb/" + s.fajl), start=1):
            if re.search(minta, tisztit(oldal.get_text())):
                talalt.add((s.jelentes, i))
    return talalt


def van_jo_talalat(talalatok, vart_oldalak):
    """Igaz, ha a találatok között van olyan darab, amely a várt oldalak egyikéről jön."""
    return any((d["jelentes"], d["pdf_oldal"]) in vart_oldalak for _, d in talalatok)

In [ ]:
# Öt jelentés: a 4. alkalom négye, és a 2024. márciusi (a mintaprojekt kérdése miatt).
JELENTESEK = ["2023-12", "2024-03", "2024-09", "2025-03", "2026-09"]

osszes_darab = darabok_betoltese()      # a 3. alkalom darabjai; ha nincs meg a fájl, kb. 10 mp
darabok = [d for d in osszes_darab if d["jelentes"] in JELENTESEK]
print(f"{len(darabok)} darab a kiválasztott {len(JELENTESEK)} jelentésben")

index = index_betoltese(darabok)          # gyorsítótár nélkül kb. 1,5 perc
print("Az index alakja:", index.shape)

## 2. Kiindulás: a modell egyedül

A 2. alkalom kérdése, forrás nélkül:

In [ ]:
KERDES_CEL = "Mekkora az MNB inflációs célja?"

valasz_egyedul = kerdez(KERDES_CEL, rendszer="Válaszolj röviden, magyarul, egy-két mondatban.", gondolkodas="none")
print(valasz_egyedul)

A várt értéket a script olvassa ki a nyers PDF-ből:

In [ ]:
_oldal_tar = {}


def oldal_szovege(jelentes, pdf_oldal):
    """Egy PDF-oldal nyers szövege, takarítva. A darabolástól független: ezzel ellenőrzünk."""
    if (jelentes, pdf_oldal) not in _oldal_tar:
        fajl = forrasok.loc[forrasok["jelentes"] == jelentes, "fajl"].iloc[0]
        _oldal_tar[(jelentes, pdf_oldal)] = tisztit(pymupdf.open("data/mnb/" + fajl)[pdf_oldal - 1].get_text())
    return _oldal_tar[(jelentes, pdf_oldal)]


def szam_a_pdfbol(minta, jelentes):
    """A minta zárójeles részei (a keresett számok) az első olyan oldalon, ahol a minta megvan."""
    fajl = forrasok.loc[forrasok["jelentes"] == jelentes, "fajl"].iloc[0]
    for i in range(1, len(pymupdf.open("data/mnb/" + fajl)) + 1):
        talalat = re.search(minta, oldal_szovege(jelentes, i))
        if talalat:
            return talalat.groups()
    return None

In [ ]:
cel_most = szam_a_pdfbol(r"a (\d) százalékos középtávú cél", "2024-03")[0]
cel_2028 = szam_a_pdfbol(r"január 1-jétől (\d,\d) százalékban határozza meg", "2026-09")[0]
print(f"A jelentések szerint a középtávú cél {cel_most}%, 2028. január 1-jétől {cel_2028}%.")
print("A modell válaszában szerepel ezek közül valamelyik?",
      any(re.search(rf"(?<![\d,]){re.escape(s)}(?![\d,])", valasz_egyedul) for s in [cel_most, cel_2028]))

Mit mondott a modell, és mit írnak a jelentések?

## 3. A lánc lépésről lépésre

Kérdés → embedding → keresés → a `k` legjobb darab → kontextus → LLM → válasz hivatkozással. Az első négy lépés a `keres`:

In [ ]:
talalatok_cel = keres(KERDES_CEL, darabok, index)
kiir(talalatok_cel)

### Forrásmegjelölés

In [ ]:
HONAPOK = {"01": "január", "02": "február", "03": "március", "04": "április", "05": "május",
           "06": "június", "07": "július", "08": "augusztus", "09": "szeptember", "10": "október",
           "11": "november", "12": "december"}

### Feladat 3.1 – Copilottal

> Írj egy `forras_cimke(d)` függvényt, amely egy darabból (szótár: `jelentes`, `pdf_oldal`, `oldal`) ilyen forrásmegjelölést készít: „MNB Inflációs jelentés, 2025. március, 62. oldal”. Az év és a hónap a `jelentes` mezőből jöjjön (`"2025-03"`), a hónap neve a `HONAPOK` szótárból. Ha az `oldal` mező `None`, a nyomtatott oldalszám helyett ezt írja: „PDF 5. oldal”.

Olvasd el: mit csinál a kód, ha az `oldal` `None`?

In [ ]:
# Ide kerül a Copilot kódja

In [ ]:
ellenoriz_fuggveny("forras_cimke", [
    (({"jelentes": "2025-03", "pdf_oldal": 64, "oldal": 62},), "MNB Inflációs jelentés, 2025. március, 62. oldal"),
    (({"jelentes": "2023-12", "pdf_oldal": 5, "oldal": None},), "MNB Inflációs jelentés, 2023. december, PDF 5. oldal"),
])

Az én változatom (ha a 3.1 még nincs kész, ezzel megy tovább a notebook):

In [ ]:
def forras_cimke(d):
    """Forrásmegjelölés egy darabhoz, például: MNB Inflációs jelentés, 2025. március, 62. oldal."""
    ev, honap = d["jelentes"].split("-")
    hol = f'{d["oldal"]}. oldal' if d["oldal"] else f'PDF {d["pdf_oldal"]}. oldal'
    return f"MNB Inflációs jelentés, {ev}. {HONAPOK[honap]}, {hol}"

### Feladat 3.2 – hibakeresés

A Copilot első változata. Futtasd, és vesd össze a forrásmegjelöléseket a fenti `kiir` sorokkal.

In [ ]:
def kontextus_v1(talalatok):
    reszek = []
    for i, (hasonlosag, d) in enumerate(talalatok, start=1):
        cimke = forras_cimke(talalatok[0][1])
        reszek.append(f"[{i}] {cimke}\n{d['szoveg']}")
    return "\n\n".join(reszek)


for sor in kontextus_v1(talalatok_cel).splitlines():
    if sor.startswith("["):
        print(sor)

Javítsd ki: írd meg a `kontextus(talalatok)` függvényt, amely minden darab elé a **saját** forrását írja (ugyanebben az alakban: `[1] forrás`, új sor, szöveg).

In [ ]:
# Ide kerül a javítás

In [ ]:
if "kontextus" in globals():
    _k = kontextus(talalatok_cel)
    kontextus_jo = all(f"[{i}] {forras_cimke(d)}\n" in _k for i, (_, d) in enumerate(talalatok_cel, start=1))
else:
    kontextus_jo = None
ellenoriz_feltetel("kontextus_jo", lambda x: x, "minden darab előtt a saját sorszáma és forrása áll")

Az én változatom:

In [ ]:
def kontextus(talalatok):
    """A modellnek szánt források: sorszám, forrásmegjelölés, aztán a darab szövege."""
    reszek = []
    for i, (hasonlosag, d) in enumerate(talalatok, start=1):
        reszek.append(f"[{i}] {forras_cimke(d)}\n{d['szoveg']}")
    return "\n\n".join(reszek)

## 4. A rendszerprompt és a teljes lánc

Négy szabály: csak a forrásokból; hivatkozás sorszámmal; figyelj a dátumra; ha nincs benne, mondd meg.

In [ ]:
RENDSZER = """Az MNB Inflációs jelentéseiből válaszolsz közgazdász hallgatóknak, magyarul, röviden.
Szabályok:
1. Csak a megadott forrásokból válaszolj. Saját tudásból ne írj számot.
2. Minden állítás után szögletes zárójelben jelöld a forrás számát, például [2].
3. Figyelj a jelentés dátumára: a régebbi jelentés előrejelzése nem tényadat. Ha a források eltérnek, a legújabb jelentésből vett tényadatot add meg.
4. Ha a források nem tartalmazzák a választ, ezt írd: "A forrásokban nincs benne." Ilyenkor ne írj számot."""

Az üzenetlista:

In [ ]:
uzenetek_cel = [
    {"role": "system", "content": RENDSZER},
    {"role": "user", "content": f"Források:\n\n{kontextus(talalatok_cel)}\n\nKérdés: {KERDES_CEL}"},
]
print(uzenetek_cel[1]["content"][:700], "…")

### Feladat 4.1 – olvasás

A rendszerprompt + 5 darab + a kérdés hány token? Írd be a legközelebbit: `500`, `1500`, `3000` vagy `5000`.

In [ ]:
tipp_4_1 = None  # a None helyére írd a tipped

A mérés (egy rövid hívás):

In [ ]:
def tokenszam(uzenetek):
    """Hány token a kérés a helyi modell szerint. Szöveget vagy üzenetlistát is elfogad.
    Egyetlen tokennyi választ kérünk, gondolkodás nélkül, hogy gyors legyen (1-2 mp)."""
    if isinstance(uzenetek, str):
        uzenetek = [{"role": "user", "content": uzenetek}]
    chat(uzenetek, gondolkodas="none", max_tokens=1)
    return utolso_hasznalat["prompt_tokens"]

In [ ]:
tokenek_cel = tokenszam(uzenetek_cel)
print(f"rendszerprompt + 5 darab + kérdés: {tokenek_cel} token")
ellenoriz("tipp_4_1", vart=min([500, 1500, 3000, 5000], key=lambda x: abs(x - tokenek_cel)))

A teljes lánc egy függvényben. Minden hívás előtt megméri és kiírja a kérés tokenszámát; ha 3000 fölött van, a leggyengébb találatot elhagyja.

In [ ]:
TOKEN_HATAR = 3000   # a 4096 tokenes ablakból ennyi a kérés; a többi a gondolkodásé és a válaszé


def rag_valasz(kerdes, darabok, index, k=5, rendszer=RENDSZER, gondolkodas="none"):
    """A teljes RAG-lánc: keresés → kontextus → tokenszám → modell. Szótárat ad vissza."""
    talalatok = keres(kerdes, darabok, index, k)
    while True:
        uzenetek = [{"role": "system", "content": rendszer},
                    {"role": "user", "content": f"Források:\n\n{kontextus(talalatok)}\n\nKérdés: {kerdes}"}]
        tokenek = tokenszam(uzenetek)
        if tokenek <= TOKEN_HATAR or len(talalatok) == 1:
            break
        talalatok = talalatok[:-1]           # nem fér bele: a leggyengébb találat kimarad
    print(f"A kérés: {tokenek} token, {len(talalatok)} darab (határ: {TOKEN_HATAR}, ablak: 4096)")
    kezdes = time.perf_counter()
    valasz = chat(uzenetek, gondolkodas=gondolkodas)["content"]
    return {"kerdes": kerdes, "valasz": valasz, "talalatok": talalatok, "tokenek": tokenek,
            "valasz_tokenek": utolso_hasznalat.get("completion_tokens"),
            "ido": time.perf_counter() - kezdes}

In [ ]:
eredmeny_cel = rag_valasz(KERDES_CEL, darabok, index)
print(eredmeny_cel["valasz"])

### Feladat 4.2 – Copilottal

> Írj egy `hivatkozasok(valasz)` függvényt, amely kigyűjti a válaszból a szögletes zárójelben álló forrásszámokat, és egész számok listájaként adja vissza, növekvő sorrendben, ismétlés nélkül. Egy zárójelben több szám is lehet vesszővel elválasztva, például `[2, 3]`. A zárójelen kívüli számokat (például 3,7 vagy 2024) hagyja figyelmen kívül.

Olvasd el: mit kezd a kód a `[2, 3]` alakkal? És a „3,7%”-kal?

In [ ]:
# Ide kerül a Copilot kódja

In [ ]:
ellenoriz_fuggveny("hivatkozasok", [
    (("Az infláció 2024-ben 3,7% volt [2].",), [2]),
    (("A cél 3% [2, 3], 2028-tól 2,5% [1].",), [1, 2, 3]),
    (("A forrásokban nincs benne.",), []),
    (("Lásd [4] és [1], illetve újra [4].",), [1, 4]),
])

Az én változatom, és a kiírás a teljes forrásokkal:

In [ ]:
def hivatkozasok(valasz):
    """A válaszban szögletes zárójelben álló forrásszámok, növekvő sorrendben, ismétlés nélkül."""
    szamok = set()
    for belseje in re.findall(r"\[([\d,\s]+)\]", valasz):     # [2] vagy [2, 3]
        szamok.update(int(x) for x in re.findall(r"\d+", belseje))
    return sorted(szamok)


def kiir_valasz(eredmeny):
    """A válasz, alatta a hivatkozott források teljes megjelölésével (jelentés, nyomtatott oldal)."""
    print(eredmeny["valasz"])
    print("\nHivatkozott források:")
    for n in hivatkozasok(eredmeny["valasz"]):
        if 1 <= n <= len(eredmeny["talalatok"]):
            print(f"  [{n}] {forras_cimke(eredmeny['talalatok'][n - 1][1])}")
        else:
            print(f"  [{n}] nincs ilyen sorszámú forrás: a modell kitalálta")

In [ ]:
kiir_valasz(eredmeny_cel)

### Ellenőrzés a nyers PDF-ből

A szám benne van a válaszban, és a **hivatkozott** oldal nyers PDF-szövegében is?

In [ ]:
def szam_benne(szam, szoveg):
    """Önálló számként szerepel-e a szám a szövegben (a 13,7-ben nincs benne a 3,7). A tizedespont is jó."""
    szoveg = re.sub(r"(\d)\.(\d)", r"\1,\2", szoveg)
    return re.search(rf"(?<![\d,]){re.escape(szam)}(?!\d|,\d)", szoveg) is not None


def alatamasztva(eredmeny, szam):
    """A szám benne van a válaszban, ÉS legalább egy hivatkozott darab nyers PDF-oldalán is."""
    if not szam_benne(szam, eredmeny["valasz"]):
        return False
    for n in hivatkozasok(eredmeny["valasz"]):
        if 1 <= n <= len(eredmeny["talalatok"]):
            d = eredmeny["talalatok"][n - 1][1]
            if szam_benne(szam, oldal_szovege(d["jelentes"], d["pdf_oldal"])):
                return True
    return False

In [ ]:
ellenoriz_feltetel("eredmeny_cel", lambda e: alatamasztva(e, cel_2028),
                   f"a válaszban szerepel a {cel_2028}%, és a hivatkozott oldalon is ott áll")

### A 4. alkalom kérdése: régi előrejelzés vagy tényadat?

In [ ]:
KERDES_2024 = "Mekkora volt az infláció 2024-ben?"
eredmeny_2024 = rag_valasz(KERDES_2024, darabok, index)
kiir(eredmeny_2024["talalatok"])
print()
kiir_valasz(eredmeny_2024)

In [ ]:
vart_2024 = szam_a_pdfbol(r"2024-ben az infláció (\d,\d) százalékon alakult", "2025-03")[0]
print("A 2025. márciusi jelentés szerint:", vart_2024)
ellenoriz_feltetel("eredmeny_2024", lambda e: alatamasztva(e, vart_2024),
                   f"a válaszban a tényadat ({vart_2024}%) áll, és a hivatkozott oldalon is ott van")

Milyen számok vannak az öt találatban? Melyiket választotta a modell, és melyik jelentésre hivatkozott? Olvasd el a válasz minden mondatát: van benne olyan, amit a script nem ellenőriz?

## 5. A RAG mint kontextustervezés

Ugyanaz a modell, ugyanaz az öt darab, kétféle kontextus. Először csak a darabok szövege, forrás és rendszerprompt nélkül:

In [ ]:
KERDES_MARC = "Mekkora inflációt várt az MNB 2024-re a 2024. márciusi Inflációs jelentésben?"
talalatok_marc = keres(KERDES_MARC, darabok, index)
kiir(talalatok_marc)

nyers_kerdes = "\n\n".join(d["szoveg"] for _, d in talalatok_marc) + "\n\nKérdés: " + KERDES_MARC
print(f"\nA kérés: {tokenszam(nyers_kerdes)} token")
valasz_nyers = kerdez(nyers_kerdes, gondolkodas="none")
print(valasz_nyers)

Ugyanez forrásmegjelöléssel és rendszerprompttal:

In [ ]:
eredmeny_marc = rag_valasz(KERDES_MARC, darabok, index)
kiir_valasz(eredmeny_marc)

In [ ]:
vart_marc = szam_a_pdfbol(r"Az infláció 2024-ben (\d,\d)–(\d,\d) százalék", "2024-03")
print("A 2024. márciusi jelentés szerint:", "–".join(vart_marc), "százalék")
ellenoriz_feltetel("eredmeny_marc", lambda e: all(alatamasztva(e, s) for s in vart_marc),
                   "a sáv mindkét széle a válaszban, és a hivatkozott oldalon is")

Mi változott a két hívás között, és mi nem?

## 6. Ha nincs benne: „A forrásokban nincs benne.”

> Mennyi volt az átlagos albérletár Debrecenben 2024-ben?

Előbb a script ellenőrzi, hogy a gyűjteményben tényleg nincs rá válasz:

In [ ]:
KERDES_NINCS = "Mennyi volt az átlagos albérletár Debrecenben 2024-ben?"

debrecen = oldalak_ahol(r"Debrecen", JELENTESEK)
alberlet = oldalak_ahol(r"albérlet|lakbér|bérleti díj", JELENTESEK)
print("Oldalak, ahol Debrecen szerepel:", sorted(debrecen))
print("Oldalak, ahol albérlet, lakbér vagy bérleti díj szerepel:", sorted(alberlet))
print("Közös oldal:", sorted(debrecen & alberlet) or "nincs")

A feltételt a hívás **előtt** írjuk meg.

### Feladat 6.1 – Copilottal

> Írj egy `nem_tudja(valasz)` függvényt, amely `True`-t ad, ha a válasz jelzi, hogy a forrásokban nincs meg a válasz (szerepel benne kisbetűsítve a „nincs benne”, a „nem tudom”, a „nem tartalmaz” vagy a „nem szerepel”), ÉS nincs benne szám. Az évszámok (négyjegyű, 19-cel vagy 20-szal kezdődő számok) nem számítanak számnak.

Olvasd el: mit ad a kód egy olyan válaszra, amely azt mondja, nincs benne, de utána mégis tippel egy összeget?

In [ ]:
# Ide kerül a Copilot kódja

In [ ]:
ellenoriz_fuggveny("nem_tudja", [
    (("A forrásokban nincs benne.",), True),
    (("Az infláció 2024-ben 3,7% volt [2].",), False),
    (("A forrásokban nincs benne, de kb. 150 ezer forint lehetett.",), False),
    (("Nem tudom: a források 2024-ről ilyet nem írnak.",), True),
])

Az én változatom, és a hívás:

In [ ]:
def nem_tudja(valasz):
    """A válasz jelzi, hogy a forrásokban nincs benne, és nem ad meg számot (évszámot kivéve)."""
    jelzi = re.search(r"nincs benne|nem tudom|nem tartalmaz|nem szerepel", valasz.lower()) is not None
    szamok = [s for s in re.findall(r"\d+(?:[,.]\d+)?", valasz) if not re.fullmatch(r"(19|20)\d\d", s)]
    return jelzi and not szamok

In [ ]:
eredmeny_nincs = rag_valasz(KERDES_NINCS, darabok, index)
kiir(eredmeny_nincs["talalatok"])
print()
kiir_valasz(eredmeny_nincs)

In [ ]:
ellenoriz_feltetel("eredmeny_nincs", lambda e: nem_tudja(e["valasz"]),
                   "a modell jelezte, hogy a forrásokban nincs benne, és nem talált ki számot")

Milyen darabokat talált a keresés, és mekkora a hasonlóságuk? Ki döntötte el, hogy ez nem válasz?

## 7. A költségvetés gondolkodással

Ugyanaz a kérdés gondolkodással. **Kb. 1 perc.**

In [ ]:
eredmeny_gond = rag_valasz(KERDES_2024, darabok, index, gondolkodas=None)
kiir_valasz(eredmeny_gond)

kolts = pd.DataFrame([
    {"hívás": "gondolkodás nélkül", "kérés (token)": eredmeny_2024["tokenek"],
     "gondolkodás + válasz (token)": eredmeny_2024["valasz_tokenek"], "idő (mp)": round(eredmeny_2024["ido"])},
    {"hívás": "gondolkodással", "kérés (token)": eredmeny_gond["tokenek"],
     "gondolkodás + válasz (token)": eredmeny_gond["valasz_tokenek"], "idő (mp)": round(eredmeny_gond["ido"])},
]).set_index("hívás")
kolts["összesen (a 4096-ból)"] = kolts["kérés (token)"] + kolts["gondolkodás + válasz (token)"]
kolts

In [ ]:
ellenoriz_feltetel("eredmeny_gond", lambda e: e["tokenek"] + e["valasz_tokenek"] < 4096,
                   "a kérés, a gondolkodás és a válasz együtt belefért a 4096 tokenes ablakba")

Mennyi tokent vitt el a gondolkodás, és mennyi időt? Miért 3000 a kérés határa?

## 8. A projektben

A RAG-lánc a saját gyűjteményeden fut (párban: a közös gyűjteményeteken). Ami változik: `forras_cimke`, a rendszerprompt első mondata és 3. szabálya, a `JELENTESEK` és a gyorsítótár neve. Az ellenőrzés elve ugyanaz.

### Otthonra

Tedd fel a 4. alkalom három saját kérdését a `rag_valasz` függvénynek, és nézd meg mindegyiknél:

1. hivatkozik a modell? (`hivatkozasok`)
2. a hivatkozott oldalon tényleg ott a szám? Ehhez keresd ki a PDF-ben a mondatot, és írj rá egy `szam_a_pdfbol` mintát, aztán `alatamasztva`.
3. ha a kérdésre nincs válasz a jelentésekben, azt mondja-e, hogy nincs benne?

Ezek lesznek a 6. alkalom tesztkérdéseinek első darabjai.

In [ ]:
# A saját kérdéseid:
